# Haaste: Tekstin analysointi Data Sciencestä

Tässä esimerkissä tehdään yksinkertainen harjoitus, joka kattaa kaikki perinteisen data science -prosessin vaiheet. Sinun ei tarvitse kirjoittaa koodia, voit vain klikata alla olevia soluja suorittaaksesi ne ja tarkkailla tulosta. Haasteena sinua kannustetaan kokeilemaan tätä koodia eri aineistolla.

## Tavoite

Tässä oppitunnissa olemme käsitelleet erilaisia data scienceen liittyviä käsitteitä. Yritetään löytää lisää aiheeseen liittyviä käsitteitä tekemällä **tekstinlouhintaa**. Aloitamme Data Science -teemaisella tekstillä, poimimme siitä avainsanoja ja yritämme sitten visualisoida tuloksen.

Tekstinä käytän Wikipedia-sivua Data Sciencestä:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Vaihe 1: Datan hankinta

Ensimmäinen vaihe kaikessa datatieteessä on datan hankinta. Käytämme siihen `requests`-kirjastoa:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Vaihe 2: Tietojen muuntaminen

Seuraava vaihe on muuntaa tiedot käsittelyyn sopivaan muotoon. Meidän tapauksessamme olemme ladanneet HTML-lähdekoodin sivulta, ja meidän täytyy muuttaa se tavalliseksi tekstiksi.

Tämä voidaan tehdä monella tavalla. Käytämme [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), suositeltua Python-kirjastoa HTML:n jäsentämiseen. BeautifulSoupin avulla voimme kohdistaa tietyt HTML-elementit, jolloin voimme keskittyä Wikipedian pääartikkelin sisältöön ja vähentää navigointivalikoita, sivupalkkeja, alatunnisteita ja muuta epäolennaista sisältöä (vaikka osa boilerplate-tekstistä saattaa jäädä jäljelle).


Ensin meidän täytyy asentaa BeautifulSoup-kirjasto HTML:n jäsentämistä varten:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Vaihe 3: Näkemyksen saaminen

Tärkein vaihe on muuntaa datamme johonkin muotoon, josta voimme saada näkemyksiä. Tässä tapauksessa haluamme poimia avainsanoja tekstistä ja nähdä, mitkä avainsanat ovat merkityksellisempiä.

Käytämme Python-kirjastoa nimeltä [RAKE](https://github.com/aneesha/RAKE) avainsanojen poimintaan. Ensin asennetaan tämä kirjasto, jos sitä ei vielä ole: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Pääk toiminnallisuus on saatavilla `Rake`-objektista, jota voimme mukauttaa käyttämällä joitakin parametreja. Tässä tapauksessa asetamme avainsanan vähimmäispituudeksi 5 merkkiä, avainsanan vähimmäistiheydeksi dokumentissa 3 ja avainsanan maksimisanamääräksi 2. Voit vapaasti kokeilla muita arvoja ja tarkkailla tulosta.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Saamme luettelon termeistä yhdessä niiden merkityksen asteen kanssa. Kuten näet, merkittävimmät alat, kuten koneoppiminen ja big data, ovat listalla ylimmillä sijoilla.

## Vaihe 4: Tuloksen visualisointi

Ihmiset pystyvät parhaiten tulkitsemaan tietoa visuaalisessa muodossa. Siksi usein on mielekästä visualisoida dataa saadakseen oivalluksia. Voimme käyttää Pythonin `matplotlib`-kirjastoa piirtämään yksinkertaisen jakautuman avainsanoista niiden merkityksen mukaan:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

On kuitenkin olemassa vielä parempi tapa visualisoida sanatiheyksiä – käyttämällä **Word Cloud** -pilveä. Meidän täytyy asentaa toinen kirjasto piirtämään sanapilvi avainsanalistastamme.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud`-olio vastaa joko alkuperäisen tekstin tai ennakkoon laskettujen sanojen ja niiden esiintymistiheyksien ottamisesta, ja palauttaa kuvan, joka voidaan sitten näyttää käyttäen `matplotlib`-kirjastoa:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Voimme myös syöttää alkuperäisen tekstin `WordCloud`-luokalle – katsotaan, saammeko samanlaisen tuloksen:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Voit nähdä, että sanapilvi näyttää nyt vaikuttavammalta, mutta se sisältää myös paljon häiriötekijöitä (esim. asiaankuulumattomia sanoja kuten `Retrieved on`). Lisäksi saamme vähemmän avainsanoja, jotka koostuvat kahdesta sanasta, kuten *data scientist* tai *computer science*. Tämä johtuu siitä, että RAKE-algoritmi valitsee tekstistä avainsanoja paljon paremmin. Tämä esimerkki havainnollistaa esikäsittelyn ja datan puhdistuksen merkitystä, koska selkeä kuva lopuksi antaa meille mahdollisuuden tehdä parempia päätöksiä.

Tässä harjoituksessa olemme käyneet läpi yksinkertaisen prosessin, jossa olemme poimineet merkitystä Wikipedian tekstistä avainsanojen ja sanapilven muodossa. Tämä esimerkki on varsin yksinkertainen, mutta se demonstroi hyvin kaikki ne tyypilliset vaiheet, joita data scientist ottaa työskennellessään datan kanssa alkaen datan hankinnasta ja päätyen visualisointiin.

Kurssillamme käsittelemme kaikkia näitä vaiheita yksityiskohtaisesti.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
